# 📊 Propensity Modeling - Exploratory Data Analysis (EDA)

Notebook này phục vụ phân tích khám phá dữ liệu (EDA) ban đầu:
1. **Tổng quan dữ liệu**: Kích thước, kiểu dữ liệu, tỷ lệ giá trị khuyết (missing).
2. **Phân phối biến mục tiêu (Target)**: Kiểm tra độ mất cân bằng lớp (Class Imbalance).
3. **Phân tích biến số (Numerical Features)**: Phân phối, outlier, tương quan với target.
4. **Phân tích biến phân loại (Categorical Features)**: Tỷ lệ chuyển đổi (conversion rate) theo từng nhóm.
5. **Đề xuất Feature Engineering**: Tạo biến mới cho pipeline chính thức.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Thêm đường dẫn src vào system path
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_config, load_data, validate_schema

# Cấu hình hiển thị đồ thị
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 50)

## 1. Nạp cấu hình và dữ liệu thô

In [ ]:
config = load_config(PROJECT_ROOT / "configs/config.yaml")
raw_data_path = PROJECT_ROOT / config["paths"]["raw_data"]

print(f"Đường dẫn dữ liệu thô: {raw_data_path}")
if raw_data_path.exists():
    df = load_data(raw_data_path)
    print("Shape:", df.shape)
    display(df.head())
else:
    print("⚠️ Chưa có file dữ liệu thô. Vui lòng thêm file vào data/raw/!")

## 2. Kiểm tra chất lượng dữ liệu (Data Quality & Missing Values)

In [ ]:
if 'df' in locals():
    # Thống kê missing values
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
    display(missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing %', ascending=False))
    
    # Thống kê trùng lặp ID
    id_col = config["schema"]["id_column"]
    if id_col in df.columns:
        print(f"Trùng lặp ID ({id_col}):", df.duplicated(subset=[id_col]).sum())

## 3. Phân tích biến mục tiêu (Target Imbalance)

In [ ]:
if 'df' in locals():
    target_col = config["schema"]["target_column"]
    if target_col in df.columns:
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        df[target_col].value_counts().plot(kind='bar', ax=ax[0], color=['steelblue', 'coral'])
        ax[0].set_title("Số lượng nhãn mục tiêu")
        
        df[target_col].value_counts(normalize=True).plot(kind='pie', autopct='%1.2f%%', ax=ax[1], colors=['steelblue', 'coral'])
        ax[1].set_ylabel('')
        ax[1].set_title("Tỷ lệ % chuyển đổi")
        plt.tight_layout()
        plt.show()

## 4. Phân tích tương quan & đặc trưng số (Numerical Features)

In [ ]:
if 'df' in locals():
    num_cols = config["schema"]["numerical_features"]
    valid_num_cols = [c for c in num_cols if c in df.columns]
    if valid_num_cols:
        display(df[valid_num_cols].describe().T)
        
        # Correlation Heatmap
        plt.figure(figsize=(10, 8))
        sns.heatmap(df[valid_num_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
        plt.title("Correlation Heatmap giữa các biến số")
        plt.show()

## 5. Phân tích biến phân loại (Categorical Features vs Target)

In [ ]:
if 'df' in locals():
    cat_cols = config["schema"]["categorical_features"]
    target_col = config["schema"]["target_column"]
    
    for col in cat_cols:
        if col in df.columns and target_col in df.columns:
            summary = df.groupby(col)[target_col].agg(['count', 'mean']).reset_index()
            summary.columns = [col, 'Total Customers', 'Conversion Rate']
            print(f"\n--- {col} ---")
            display(summary.sort_values(by='Conversion Rate', ascending=False))